<a href="https://colab.research.google.com/github/fajriantomanungki/lomba_kti/blob/main/Data/Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas requests openpyxl tqdm unidecode rapidfuzz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import re
import time
import json
import requests
import pandas as pd
import numpy as np

from io import BytesIO
from tqdm import tqdm
from unidecode import unidecode
from rapidfuzz import process, fuzz


BPS_API_KEY = "77be3582731278298c7d1d3bf9dff046"

BASE_URL = "https://webapi.bps.go.id/v1/api"

OUT_DIR = "/content/drive/MyDrive/Karaya Ilmiah/AEDS_Sulampua/data/bps_kemiskinan_kepadatan"
os.makedirs(OUT_DIR, exist_ok=True)

print("Output folder:", OUT_DIR)

Output folder: /content/drive/MyDrive/Karaya Ilmiah/AEDS_Sulampua/data/bps_kemiskinan_kepadatan


In [4]:
data_text = """
provinsi,kabupaten_kota
Gorontalo,Boalemo
Gorontalo,Bonebolango
Gorontalo,Gorontalo
Gorontalo,Gorontaloutara
Gorontalo,Kotagorontalo
Gorontalo,Pohuwato
Maluku,Ambon
Maluku,Buru
Maluku,Buruselatan
Maluku,Kepulauanaru
Maluku,Malukubaratdaya
Maluku,Malukutengah
Maluku,Malukutenggara
Maluku,Malukutenggarabarat
Maluku,Serambagianbarat
Maluku,Serambagiantimur
Maluku,Tual
Malukuutara,Halmaherabarat
Malukuutara,Halmaheraselatan
Malukuutara,Halmaheratengah
Malukuutara,Halmaheratimur
Malukuutara,Halmaherautara
Malukuutara,Kepulauansula
Malukuutara,Pulaumorotai
Malukuutara,Ternate
Malukuutara,Tidorekepulauan
Papua,Asmat
Papua,Biaknumfor
Papua,Bovendigoel
Papua,Deiyai
Papua,Dogiyai
Papua,Intanjaya
Papua,Jayapura
Papua,Jayawijaya
Papua,Keerom
Papua,Kepulauanyapen
Papua,Kotajayapura
Papua,Lannyjaya
Papua,Mamberamoraya
Papua,Mamberamotengah
Papua,Mappi
Papua,Merauke
Papua,Mimika
Papua,Nabire
Papua,Nduga
Papua,Paniai
Papua,Pegununganbintang
Papua,Puncak
Papua,Puncakjaya
Papua,Sarmi
Papua,Supiori
Papua,Tolikara
Papua,Waropen
Papua,Yahukimo
Papua,Yalimo
Papuabarat,Fakfak
Papuabarat,Kaimana
Papuabarat,Kotasorong
Papuabarat,Manokwari
Papuabarat,Maybrat
Papuabarat,Rajaampat
Papuabarat,Sorong
Papuabarat,Sorongselatan
Papuabarat,Tambrauw
Papuabarat,Telukbintuni
Papuabarat,Telukwondama
Sulawesibarat,Majene
Sulawesibarat,Mamasa
Sulawesibarat,Mamuju
Sulawesibarat,Mamujuutara
Sulawesibarat,Polewalimandar
Sulawesiselatan,Bantaeng
Sulawesiselatan,Barru
Sulawesiselatan,Bone
Sulawesiselatan,Bulukumba
Sulawesiselatan,Enrekang
Sulawesiselatan,Gowa
Sulawesiselatan,Jeneponto
Sulawesiselatan,Kepulauanselayar
Sulawesiselatan,Luwu
Sulawesiselatan,Luwutimur
Sulawesiselatan,Luwuutara
Sulawesiselatan,Makassar
Sulawesiselatan,Maros
Sulawesiselatan,Palopo
Sulawesiselatan,Pangkajenedankepulauan
Sulawesiselatan,Parepare
Sulawesiselatan,Pinrang
Sulawesiselatan,Sidenrengrappang
Sulawesiselatan,Sinjai
Sulawesiselatan,Soppeng
Sulawesiselatan,Takalar
Sulawesiselatan,Tanatoraja
Sulawesiselatan,Torajautara
Sulawesiselatan,Wajo
Sulawesitengah,Banggai
Sulawesitengah,Banggaikepulauan
Sulawesitengah,Buol
Sulawesitengah,Donggala
Sulawesitengah,Morowali
Sulawesitengah,Palu
Sulawesitengah,Parigimoutong
Sulawesitengah,Poso
Sulawesitengah,Sigi
Sulawesitengah,Tojouna-Una
Sulawesitengah,Toli-Toli
Sulawesitenggara,Bau-Bau
Sulawesitenggara,Bombana
Sulawesitenggara,Buton
Sulawesitenggara,Butonutara
Sulawesitenggara,Kendari
Sulawesitenggara,Kolaka
Sulawesitenggara,Kolakautara
Sulawesitenggara,Konawe
Sulawesitenggara,Konaweselatan
Sulawesitenggara,Konaweutara
Sulawesitenggara,Muna
Sulawesitenggara,Wakatobi
Sulawesiutara,Bitung
Sulawesiutara,Bolaangmongondow
Sulawesiutara,Bolaangmongondowselatan
Sulawesiutara,Bolaangmongondowtimur
Sulawesiutara,Bolaangmongondowutara
Sulawesiutara,Kepulauansangihe
Sulawesiutara,Kepulauantalaud
Sulawesiutara,Kotamobagu
Sulawesiutara,Manado
Sulawesiutara,Minahasa
Sulawesiutara,Minahasaselatan
Sulawesiutara,Minahasatenggara
Sulawesiutara,Minahasautara
Sulawesiutara,Siautagulandangbiaro
Sulawesiutara,Tomohon
"""

from io import StringIO

target = pd.read_csv(StringIO(data_text))
target.head(), target.shape

(    provinsi  kabupaten_kota
 0  Gorontalo         Boalemo
 1  Gorontalo     Bonebolango
 2  Gorontalo       Gorontalo
 3  Gorontalo  Gorontaloutara
 4  Gorontalo   Kotagorontalo,
 (133, 2))

In [5]:
def norm_name(x):
    """
    Normalisasi nama untuk matching:
    - huruf kecil
    - hilangkan aksen
    - hilangkan spasi, titik, koma, tanda hubung
    - hilangkan awalan kabupaten/kota/kab/kota administrasi
    """
    if pd.isna(x):
        return ""
    x = str(x)
    x = unidecode(x)
    x = x.lower()

    # hilangkan awalan umum
    x = re.sub(r"\bkabupaten\b", "", x)
    x = re.sub(r"\bkab\.\b", "", x)
    x = re.sub(r"\bkab\b", "", x)
    x = re.sub(r"\bkota administrasi\b", "", x)
    x = re.sub(r"\bkota\b", "", x)

    # ejaan lama/variasi
    x = x.replace("kep.", "kepulauan")
    x = x.replace("kep ", "kepulauan ")

    # khusus variasi umum
    replacements = {
        "tojo una una": "tojounauna",
        "tojo una-una": "tojounauna",
        "tojo unauna": "tojounauna",
        "toli toli": "tolitoli",
        "toli-toli": "tolitoli",
        "bau bau": "baubau",
        "bau-bau": "baubau",
        "bone bolango": "bonebolango",
        "gorontalo utara": "gorontaloutara",
        "kota gorontalo": "kotagorontalo",
        "maluku barat daya": "malukubaratdaya",
        "maluku tengah": "malukutengah",
        "maluku tenggara": "malukutenggara",
        "seram bagian barat": "serambagianbarat",
        "seram bagian timur": "serambagiantimur",
        "halmahera barat": "halmaherabarat",
        "halmahera selatan": "halmaheraselatan",
        "halmahera tengah": "halmaheratengah",
        "halmahera timur": "halmaheratimur",
        "halmahera utara": "halmaherautara",
        "tidore kepulauan": "tidorekepulauan",
        "biak numfor": "biaknumfor",
        "boven digoel": "bovendigoel",
        "intan jaya": "intanjaya",
        "kepulauan yapen": "kepulauanyapen",
        "mamberamo raya": "mamberamoraya",
        "mamberamo tengah": "mamberamotengah",
        "pegunungan bintang": "pegununganbintang",
        "puncak jaya": "puncakjaya",
        "raja ampat": "rajaampat",
        "sorong selatan": "sorongselatan",
        "teluk bintuni": "telukbintuni",
        "teluk wondama": "telukwondama",
        "mamuju utara": "mamujuutara",
        "polewali mandar": "polewalimandar",
        "kepulauan selayar": "kepulauanselayar",
        "luwu timur": "luwutimur",
        "luwu utara": "luwuutara",
        "pangkajene dan kepulauan": "pangkajenedankepulauan",
        "sidenreng rappang": "sidenrengrappang",
        "tana toraja": "tanatoraja",
        "toraja utara": "torajautara",
        "banggai kepulauan": "banggaikepulauan",
        "parigi moutong": "parigimoutong",
        "buton utara": "butonutara",
        "kolaka utara": "kolakautara",
        "konawe selatan": "konaweselatan",
        "konawe utara": "konaweutara",
        "bolaang mongondow": "bolaangmongondow",
        "bolaang mongondow selatan": "bolaangmongondowselatan",
        "bolaang mongondow timur": "bolaangmongondowtimur",
        "bolaang mongondow utara": "bolaangmongondowutara",
        "kepulauan sangihe": "kepulauansangihe",
        "kepulauan talaud": "kepulauantalaud",
        "minahasa selatan": "minahasaselatan",
        "minahasa tenggara": "minahasatenggara",
        "minahasa utara": "minahasautara",
        "siau tagulandang biaro": "siautagulandangbiaro",
    }

    for a, b in replacements.items():
        x = x.replace(a, b)

    x = re.sub(r"[^a-z0-9]", "", x)
    return x

target["kabupaten_kota_norm"] = target["kabupaten_kota"].apply(norm_name)
target["provinsi_norm"] = target["provinsi"].apply(norm_name)

target.head()

,provinsi,kabupaten_kota,kabupaten_kota_norm,provinsi_norm
0,Gorontalo,Boalemo,boalemo,gorontalo
1,Gorontalo,Bonebolango,bonebolango,gorontalo
2,Gorontalo,Gorontalo,gorontalo,gorontalo
3,Gorontalo,Gorontaloutara,gorontaloutara,gorontalo
4,Gorontalo,Kotagorontalo,kotagorontalo,gorontalo


In [7]:
# ============================================================
# MAPPING DOMAIN BPS KAB/KOTA SULAMPUA
# Dipakai untuk menghindari error HTTP 403 pada endpoint /domain
# ============================================================

domain_manual = [
    # Gorontalo
    ("Gorontalo", "Boalemo", "7501"),
    ("Gorontalo", "Gorontalo", "7502"),
    ("Gorontalo", "Pohuwato", "7503"),
    ("Gorontalo", "Bonebolango", "7504"),
    ("Gorontalo", "Gorontaloutara", "7505"),
    ("Gorontalo", "Kotagorontalo", "7571"),

    # Maluku
    ("Maluku", "Malukutenggarabarat", "8101"),
    ("Maluku", "Malukutenggara", "8102"),
    ("Maluku", "Malukutengah", "8103"),
    ("Maluku", "Buru", "8104"),
    ("Maluku", "Kepulauanaru", "8105"),
    ("Maluku", "Serambagianbarat", "8106"),
    ("Maluku", "Serambagiantimur", "8107"),
    ("Maluku", "Malukubaratdaya", "8108"),
    ("Maluku", "Buruselatan", "8109"),
    ("Maluku", "Ambon", "8171"),
    ("Maluku", "Tual", "8172"),

    # Maluku Utara
    ("Malukuutara", "Halmaherabarat", "8201"),
    ("Malukuutara", "Halmaheratengah", "8202"),
    ("Malukuutara", "Kepulauansula", "8203"),
    ("Malukuutara", "Halmaheraselatan", "8204"),
    ("Malukuutara", "Halmaherautara", "8205"),
    ("Malukuutara", "Halmaheratimur", "8206"),
    ("Malukuutara", "Pulaumorotai", "8207"),
    ("Malukuutara", "Taliabu", "8208"),
    ("Malukuutara", "Ternate", "8271"),
    ("Malukuutara", "Tidorekepulauan", "8272"),

    # Papua / Papua Selatan / Papua Tengah / Papua Pegunungan
    # Domain historis BPS masih memakai kode 91xx untuk banyak kabupaten lama
    ("Papua", "Merauke", "9101"),
    ("Papua", "Jayawijaya", "9102"),
    ("Papua", "Jayapura", "9103"),
    ("Papua", "Nabire", "9104"),
    ("Papua", "Kepulauanyapen", "9105"),
    ("Papua", "Biaknumfor", "9106"),
    ("Papua", "Paniai", "9107"),
    ("Papua", "Puncakjaya", "9108"),
    ("Papua", "Mimika", "9109"),
    ("Papua", "Bovendigoel", "9110"),
    ("Papua", "Mappi", "9111"),
    ("Papua", "Asmat", "9112"),
    ("Papua", "Yahukimo", "9113"),
    ("Papua", "Pegununganbintang", "9114"),
    ("Papua", "Tolikara", "9115"),
    ("Papua", "Sarmi", "9116"),
    ("Papua", "Keerom", "9117"),
    ("Papua", "Waropen", "9118"),
    ("Papua", "Supiori", "9119"),
    ("Papua", "Mamberamoraya", "9120"),
    ("Papua", "Nduga", "9121"),
    ("Papua", "Lannyjaya", "9122"),
    ("Papua", "Mamberamotengah", "9123"),
    ("Papua", "Yalimo", "9124"),
    ("Papua", "Puncak", "9125"),
    ("Papua", "Dogiyai", "9126"),
    ("Papua", "Intanjaya", "9127"),
    ("Papua", "Deiyai", "9128"),
    ("Papua", "Kotajayapura", "9171"),

    # Papua Barat / Papua Barat Daya
    ("Papuabarat", "Fakfak", "9201"),
    ("Papuabarat", "Kaimana", "9202"),
    ("Papuabarat", "Telukwondama", "9203"),
    ("Papuabarat", "Telukbintuni", "9204"),
    ("Papuabarat", "Manokwari", "9205"),
    ("Papuabarat", "Sorongselatan", "9206"),
    ("Papuabarat", "Sorong", "9207"),
    ("Papuabarat", "Rajaampat", "9208"),
    ("Papuabarat", "Tambrauw", "9209"),
    ("Papuabarat", "Maybrat", "9210"),
    ("Papuabarat", "Manokwariselatan", "9211"),
    ("Papuabarat", "Pegununganarfak", "9212"),
    ("Papuabarat", "Kotasorong", "9271"),

    # Sulawesi Barat
    ("Sulawesibarat", "Majene", "7601"),
    ("Sulawesibarat", "Polewalimandar", "7602"),
    ("Sulawesibarat", "Mamasa", "7603"),
    ("Sulawesibarat", "Mamuju", "7604"),
    ("Sulawesibarat", "Mamujuutara", "7605"),
    ("Sulawesibarat", "Mamuju Tengah", "7606"),

    # Sulawesi Selatan
    ("Sulawesiselatan", "Kepulauanselayar", "7301"),
    ("Sulawesiselatan", "Bulukumba", "7302"),
    ("Sulawesiselatan", "Bantaeng", "7303"),
    ("Sulawesiselatan", "Jeneponto", "7304"),
    ("Sulawesiselatan", "Takalar", "7305"),
    ("Sulawesiselatan", "Gowa", "7306"),
    ("Sulawesiselatan", "Sinjai", "7307"),
    ("Sulawesiselatan", "Maros", "7308"),
    ("Sulawesiselatan", "Pangkajenedankepulauan", "7309"),
    ("Sulawesiselatan", "Barru", "7310"),
    ("Sulawesiselatan", "Bone", "7311"),
    ("Sulawesiselatan", "Soppeng", "7312"),
    ("Sulawesiselatan", "Wajo", "7313"),
    ("Sulawesiselatan", "Sidenrengrappang", "7314"),
    ("Sulawesiselatan", "Pinrang", "7315"),
    ("Sulawesiselatan", "Enrekang", "7316"),
    ("Sulawesiselatan", "Luwu", "7317"),
    ("Sulawesiselatan", "Tanatoraja", "7318"),
    ("Sulawesiselatan", "Luwuutara", "7322"),
    ("Sulawesiselatan", "Luwutimur", "7324"),
    ("Sulawesiselatan", "Torajautara", "7326"),
    ("Sulawesiselatan", "Makassar", "7371"),
    ("Sulawesiselatan", "Parepare", "7372"),
    ("Sulawesiselatan", "Palopo", "7373"),

    # Sulawesi Tengah
    ("Sulawesitengah", "Banggai", "7201"),
    ("Sulawesitengah", "Poso", "7202"),
    ("Sulawesitengah", "Donggala", "7203"),
    ("Sulawesitengah", "Toli-Toli", "7204"),
    ("Sulawesitengah", "Buol", "7205"),
    ("Sulawesitengah", "Morowali", "7206"),
    ("Sulawesitengah", "BanggaiKepulauan", "7207"),
    ("Sulawesitengah", "Parigimoutong", "7208"),
    ("Sulawesitengah", "Tojouna-Una", "7209"),
    ("Sulawesitengah", "Sigi", "7210"),
    ("Sulawesitengah", "Banggailaut", "7211"),
    ("Sulawesitengah", "Morowaliutara", "7212"),
    ("Sulawesitengah", "Palu", "7271"),

    # Sulawesi Tenggara
    ("Sulawesitenggara", "Buton", "7401"),
    ("Sulawesitenggara", "Muna", "7402"),
    ("Sulawesitenggara", "Konawe", "7403"),
    ("Sulawesitenggara", "Kolaka", "7404"),
    ("Sulawesitenggara", "Konaweselatan", "7405"),
    ("Sulawesitenggara", "Bombana", "7406"),
    ("Sulawesitenggara", "Wakatobi", "7407"),
    ("Sulawesitenggara", "Kolakautara", "7408"),
    ("Sulawesitenggara", "Butonutara", "7409"),
    ("Sulawesitenggara", "Konaweutara", "7410"),
    ("Sulawesitenggara", "Kolakatimur", "7411"),
    ("Sulawesitenggara", "Konawekepulauan", "7412"),
    ("Sulawesitenggara", "Munabarat", "7413"),
    ("Sulawesitenggara", "Butontengah", "7414"),
    ("Sulawesitenggara", "Butonselatan", "7415"),
    ("Sulawesitenggara", "Kendari", "7471"),
    ("Sulawesitenggara", "Bau-Bau", "7472"),

    # Sulawesi Utara
    ("Sulawesiutara", "Bolaangmongondow", "7101"),
    ("Sulawesiutara", "Minahasa", "7102"),
    ("Sulawesiutara", "Kepulauansangihe", "7103"),
    ("Sulawesiutara", "Kepulauantalaud", "7104"),
    ("Sulawesiutara", "Minahasaselatan", "7105"),
    ("Sulawesiutara", "Minahasautara", "7106"),
    ("Sulawesiutara", "Bolaangmongondowutara", "7107"),
    ("Sulawesiutara", "Siautagulandangbiaro", "7108"),
    ("Sulawesiutara", "Minahasatenggara", "7109"),
    ("Sulawesiutara", "Bolaangmongondowselatan", "7110"),
    ("Sulawesiutara", "Bolaangmongondowtimur", "7111"),
    ("Sulawesiutara", "Manado", "7171"),
    ("Sulawesiutara", "Bitung", "7172"),
    ("Sulawesiutara", "Tomohon", "7173"),
    ("Sulawesiutara", "Kotamobagu", "7174"),
]

domain_manual = pd.DataFrame(
    domain_manual,
    columns=["provinsi_manual", "kabupaten_kota_manual", "domain_id"]
)

domain_manual["provinsi_norm"] = domain_manual["provinsi_manual"].apply(norm_name)
domain_manual["kabupaten_kota_norm"] = domain_manual["kabupaten_kota_manual"].apply(norm_name)

domain_manual.head()

,provinsi_manual,kabupaten_kota_manual,domain_id,provinsi_norm,kabupaten_kota_norm
0,Gorontalo,Boalemo,7501,gorontalo,boalemo
1,Gorontalo,Gorontalo,7502,gorontalo,gorontalo
2,Gorontalo,Pohuwato,7503,gorontalo,pohuwato
3,Gorontalo,Bonebolango,7504,gorontalo,bonebolango
4,Gorontalo,Gorontaloutara,7505,gorontalo,gorontaloutara


In [8]:
target["provinsi_norm"] = target["provinsi"].apply(norm_name)
target["kabupaten_kota_norm"] = target["kabupaten_kota"].apply(norm_name)

domain_match = target.merge(
    domain_manual[[
        "provinsi_norm",
        "kabupaten_kota_norm",
        "domain_id",
        "provinsi_manual",
        "kabupaten_kota_manual"
    ]],
    on=["provinsi_norm", "kabupaten_kota_norm"],
    how="left"
)

domain_match["domain_name_bps"] = domain_match["kabupaten_kota_manual"]
domain_match["match_score"] = np.where(domain_match["domain_id"].notna(), 100, 0)

domain_match = domain_match.rename(columns={
    "provinsi": "provinsi_input",
    "kabupaten_kota": "kabupaten_kota_input"
})

domain_match.to_csv(f"{OUT_DIR}/01_domain_match_sulampua_manual.csv", index=False)

print("Total target:", len(domain_match))
print("Berhasil match domain:", domain_match["domain_id"].notna().sum())
print("Belum match:", domain_match["domain_id"].isna().sum())

domain_match[domain_match["domain_id"].isna()]

Total target: 133
Berhasil match domain: 133
Belum match: 0


,provinsi_input,kabupaten_kota_input,kabupaten_kota_norm,provinsi_norm,domain_id,provinsi_manual,kabupaten_kota_manual,domain_name_bps,match_score


In [9]:
def bps_get(path, params=None, max_retry=3, sleep=2):
    if params is None:
        params = {}

    params = params.copy()
    params["key"] = BPS_API_KEY

    url = f"{BASE_URL}/{path.strip('/')}"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/json,text/plain,*/*",
        "Referer": "https://webapi.bps.go.id/",
    }

    for attempt in range(max_retry):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=90)

            if r.status_code == 200:
                try:
                    return r.json()
                except Exception:
                    print("Response bukan JSON.")
                    print(r.text[:300])
                    return None

            print(f"HTTP {r.status_code} pada attempt {attempt+1}")
            print(r.text[:200])

            # Kalau Cloudflare 403, jangan ulang terlalu banyak
            if r.status_code == 403 and "Just a moment" in r.text:
                print("Terdeteksi Cloudflare block. Endpoint ini kemungkinan diblokir dari Colab.")
                return None

        except Exception as e:
            print("Request error:", e)

        time.sleep(sleep * (attempt + 1))

    return None

In [12]:
KEYWORDS = {
    "kemiskinan": [
        "kemiskinan",
        "penduduk miskin",
        "persentase penduduk miskin",
        "jumlah penduduk miskin",
        "garis kemiskinan"
    ],
    "kepadatan_penduduk": [
        "kepadatan penduduk",
        "penduduk per km",
        "penduduk menurut luas wilayah",
        "luas wilayah jumlah penduduk"
    ]
}

catalog_rows = []

for _, row in tqdm(domain_match.iterrows(), total=len(domain_match)):
    domain_id = row["domain_id"]

    for indikator, kws in KEYWORDS.items():
        for kw in kws:
            try:
                tables = list_static_tables(domain_id, kw, max_pages=2)
                for t in tables:
                    t["indikator_target"] = indikator
                    t["provinsi_input"] = row["provinsi_input"]
                    t["kabupaten_kota_input"] = row["kabupaten_kota_input"]
                    t["domain_name_bps"] = row["domain_name_bps"]
                    catalog_rows.append(t)
            except Exception as e:
                catalog_rows.append({
                    "indikator_target": indikator,
                    "provinsi_input": row["provinsi_input"],
                    "kabupaten_kota_input": row["kabupaten_kota_input"],
                    "domain_id": domain_id,
                    "domain_name_bps": row["domain_name_bps"],
                    "search_keyword": kw,
                    "error": str(e)
                })

            time.sleep(0.2)

catalog = pd.DataFrame(catalog_rows)

# Rapikan duplicate table
if not catalog.empty:
    subset_cols = [c for c in ["domain_id", "indikator_target", "table_id", "title"] if c in catalog.columns]
    catalog = catalog.drop_duplicates(subset=subset_cols)

catalog.to_csv(f"{OUT_DIR}/02_catalog_tabel_bps_kemiskinan_kepadatan.csv", index=False)

catalog.head()

100%|██████████| 133/133 [03:59<00:00,  1.80s/it]


,indikator_target,provinsi_input,kabupaten_kota_input,domain_id,domain_name_bps,search_keyword,error
0,kemiskinan,Gorontalo,Boalemo,7501,Boalemo,kemiskinan,name 'list_static_tables' is not defined
5,kepadatan_penduduk,Gorontalo,Boalemo,7501,Boalemo,kepadatan penduduk,name 'list_static_tables' is not defined
9,kemiskinan,Gorontalo,Bonebolango,7504,Bonebolango,kemiskinan,name 'list_static_tables' is not defined
14,kepadatan_penduduk,Gorontalo,Bonebolango,7504,Bonebolango,kepadatan penduduk,name 'list_static_tables' is not defined
18,kemiskinan,Gorontalo,Gorontalo,7502,Gorontalo,kemiskinan,name 'list_static_tables' is not defined


In [11]:
domain_match[["provinsi_input", "kabupaten_kota_input", "domain_id", "domain_name_bps", "match_score"]].head()

,provinsi_input,kabupaten_kota_input,domain_id,domain_name_bps,match_score
0,Gorontalo,Boalemo,7501,Boalemo,100
1,Gorontalo,Bonebolango,7504,Bonebolango,100
2,Gorontalo,Gorontalo,7502,Gorontalo,100
3,Gorontalo,Gorontaloutara,7505,Gorontaloutara,100
4,Gorontalo,Kotagorontalo,7571,Kotagorontalo,100


In [13]:
def score_table_title(title, indikator):
    title_norm = str(title).lower()
    score = 0

    if indikator == "kemiskinan":
        positive = [
            "kemiskinan",
            "penduduk miskin",
            "persentase penduduk miskin",
            "jumlah penduduk miskin",
            "garis kemiskinan"
        ]
    else:
        positive = [
            "kepadatan penduduk",
            "penduduk per km",
            "jumlah penduduk",
            "luas wilayah"
        ]

    for p in positive:
        if p in title_norm:
            score += 20

    # prefer tabel yang langsung relevan
    if indikator == "kepadatan_penduduk":
        if "kepadatan penduduk" in title_norm:
            score += 50
    if indikator == "kemiskinan":
        if "penduduk miskin" in title_norm:
            score += 50

    # penalti kalau terlalu umum
    negative = ["publikasi", "berita", "infografis"]
    for n in negative:
        if n in title_norm:
            score -= 10

    return score


if not catalog.empty and "title" in catalog.columns:
    catalog["title_score"] = catalog.apply(
        lambda r: score_table_title(r.get("title", ""), r.get("indikator_target", "")),
        axis=1
    )

    candidates = (
        catalog
        .sort_values(["provinsi_input", "kabupaten_kota_input", "indikator_target", "title_score"], ascending=[True, True, True, False])
        .groupby(["provinsi_input", "kabupaten_kota_input", "domain_id", "domain_name_bps", "indikator_target"], as_index=False)
        .head(3)
    )
else:
    candidates = pd.DataFrame()

candidates.to_csv(f"{OUT_DIR}/03_kandidat_tabel_terbaik.csv", index=False)
candidates.head(20)

""


In [14]:
def get_excel_url(row):
    """
    Kolom link Excel dari API biasanya bernama 'excel'.
    Kadang berupa path relatif.
    """
    url = row.get("excel", None)
    if pd.isna(url) or url is None or str(url).strip() == "":
        return None

    url = str(url).strip()

    if url.startswith("http"):
        return url

    # fallback jika path relatif
    if url.startswith("/"):
        return "https://www.bps.go.id" + url

    return url


def download_excel_to_df(url):
    try:
        r = requests.get(url, timeout=90)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}"

        content = BytesIO(r.content)

        # baca semua sheet
        sheets = pd.read_excel(content, sheet_name=None, header=None)
        return sheets, None

    except Exception as e:
        return None, str(e)


def clean_number(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()
    if s in ["", "-", "–", "—", "na", "n/a", "..."]:
        return np.nan

    # buang catatan kaki
    s = re.sub(r"\(.*?\)", "", s)
    s = s.replace(" ", "")

    # format Indonesia: 1.234,56 -> 1234.56
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s and "." not in s:
        s = s.replace(",", ".")

    s = re.sub(r"[^0-9\.\-]", "", s)

    try:
        return float(s)
    except:
        return np.nan


def extract_latest_numeric_from_sheets(sheets):
    """
    Heuristik:
    - cari baris/kolom yang mengandung tahun 2015-2030
    - ambil nilai numerik pada tahun paling baru
    - fallback: ambil angka numerik terakhir yang masuk akal
    """
    records = []

    for sheet_name, df in sheets.items():
        df = df.copy()

        # jadikan string untuk scanning
        for i in range(df.shape[0]):
            row_vals = df.iloc[i].astype(str).tolist()

            # cari tahun di row
            years = []
            for j, val in enumerate(row_vals):
                found = re.findall(r"\b(20[0-3][0-9])\b", str(val))
                for y in found:
                    years.append((int(y), j))

            if years:
                latest_year, year_col = sorted(years)[-1]

                # cari angka di sekitar kanan/kiri tahun atau baris berikutnya
                candidate_values = []

                # angka di bawah kolom tahun
                for ii in range(i + 1, min(i + 20, df.shape[0])):
                    val = clean_number(df.iloc[ii, year_col])
                    if not pd.isna(val):
                        candidate_values.append((latest_year, val, sheet_name, ii, year_col))

                # angka di baris yang sama setelah tahun
                for jj in range(year_col + 1, min(year_col + 6, df.shape[1])):
                    val = clean_number(df.iloc[i, jj])
                    if not pd.isna(val):
                        candidate_values.append((latest_year, val, sheet_name, i, jj))

                records.extend(candidate_values)

    if records:
        # Ambil record dari tahun terbaru, lalu posisi paling bawah sebagai fallback
        records = sorted(records, key=lambda x: (x[0], x[3], x[4]), reverse=True)
        y, val, sheet, r, c = records[0]
        return {
            "tahun": y,
            "nilai": val,
            "sheet": sheet,
            "row_index": r,
            "col_index": c,
            "extract_method": "year_based"
        }

    # fallback: seluruh angka
    nums = []
    for sheet_name, df in sheets.items():
        for i in range(df.shape[0]):
            for j in range(df.shape[1]):
                val = clean_number(df.iloc[i, j])
                if not pd.isna(val):
                    nums.append((val, sheet_name, i, j))

    if nums:
        val, sheet, r, c = nums[-1]
        return {
            "tahun": np.nan,
            "nilai": val,
            "sheet": sheet,
            "row_index": r,
            "col_index": c,
            "extract_method": "fallback_last_numeric"
        }

    return {
        "tahun": np.nan,
        "nilai": np.nan,
        "sheet": None,
        "row_index": None,
        "col_index": None,
        "extract_method": "not_found"
    }

In [15]:
extract_rows = []

for _, row in tqdm(candidates.iterrows(), total=len(candidates)):
    excel_url = get_excel_url(row)

    base = {
        "provinsi_input": row.get("provinsi_input"),
        "kabupaten_kota_input": row.get("kabupaten_kota_input"),
        "domain_id": row.get("domain_id"),
        "domain_name_bps": row.get("domain_name_bps"),
        "indikator_target": row.get("indikator_target"),
        "table_id": row.get("table_id"),
        "title": row.get("title"),
        "updt_date": row.get("updt_date"),
        "excel_url": excel_url
    }

    if excel_url is None:
        base.update({
            "tahun": np.nan,
            "nilai": np.nan,
            "extract_method": "no_excel_url",
            "error": "Tidak ada link Excel dari API"
        })
        extract_rows.append(base)
        continue

    sheets, err = download_excel_to_df(excel_url)

    if err:
        base.update({
            "tahun": np.nan,
            "nilai": np.nan,
            "extract_method": "download_error",
            "error": err
        })
        extract_rows.append(base)
        continue

    result = extract_latest_numeric_from_sheets(sheets)
    base.update(result)
    base["error"] = None
    extract_rows.append(base)

    time.sleep(0.3)

extracted = pd.DataFrame(extract_rows)

extracted.to_csv(f"{OUT_DIR}/04_extracted_raw_kemiskinan_kepadatan.csv", index=False)
extracted.head()

0it [00:00, ?it/s]


""


In [16]:
# Ranking metode ekstraksi
method_rank = {
    "year_based": 1,
    "fallback_last_numeric": 2,
    "no_excel_url": 9,
    "download_error": 9,
    "not_found": 9
}

extracted["method_rank"] = extracted["extract_method"].map(method_rank).fillna(9)

# Pilih nilai terbaik: method terbaik, tahun terbaru, title_score tertinggi kalau ada
extracted = extracted.merge(
    catalog[["domain_id", "indikator_target", "table_id", "title_score"]].drop_duplicates(),
    on=["domain_id", "indikator_target", "table_id"],
    how="left"
)

final_long = (
    extracted
    .sort_values(
        ["provinsi_input", "kabupaten_kota_input", "indikator_target", "method_rank", "tahun", "title_score"],
        ascending=[True, True, True, True, False, False]
    )
    .groupby(["provinsi_input", "kabupaten_kota_input", "domain_id", "domain_name_bps", "indikator_target"], as_index=False)
    .head(1)
)

final_long.to_csv(f"{OUT_DIR}/05_final_long_bps_kemiskinan_kepadatan.csv", index=False)
final_long.head()

KeyError: 'extract_method'